In [1]:
pip install fastapi uvicorn pandas xgboost scikit-learn selenium joblib


   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.6 MB 4.2 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.6 MB 4.7 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.6 MB 4.4 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.6 MB 4.3 MB/s eta 0:00:02
   ------------------ --------------------- 4.5/9.6 MB 4.2 MB/s eta 0:00:02
   --------------------- ------------------ 5.2/9.6 MB 4.1 MB/s eta 0:00:02
   ------------------------- -------------- 6.0/9.6 MB 4.1 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.6 MB 4.1 MB/s eta 0:00:01
   -------------------------------- ------- 7.9/9.6 MB 4.1 MB/s eta 0:00:01
   ------------------------------------ --- 8.7/9.6 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.6 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 3.8 MB/s  0:00:02
   ------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# ---------------------------
# 1. LOAD DATA
# ---------------------------
labels_df = pd.read_csv(r"C:\Users\lakshita rawat\Downloads\revenge\label.csv", dtype={"id": str})
split_df = pd.read_csv(r"C:\Users\lakshita rawat\Downloads\revenge\split.csv", dtype={"id": str})
user_df = pd.read_json(r"C:\Users\lakshita rawat\Downloads\revenge\user.json")

# Merge datasets
df = labels_df.merge(split_df, on="id", how="inner")
df = df.merge(user_df, on="id", how="inner")

df.fillna(0, inplace=True)

In [3]:
# ---------------------------
# 2. CONVERT LABELS
# ---------------------------
df["label"] = df["label"].map({
    "human": 0,
    "bot": 1
})

print(df["label"].value_counts())

label
0    860057
1    139943
Name: count, dtype: int64


In [4]:
df = labels_df.merge(split_df, on="id", how="inner")
df = df.merge(user_df, on="id", how="inner")

df.fillna(0, inplace=True)

print(df.shape)
print(df["split"].value_counts())
print(df["label"].value_counts())

(1000000, 16)
split
train    700000
val      200000
test     100000
Name: count, dtype: int64
label
human    860057
bot      139943
Name: count, dtype: int64


In [5]:
import json
import pandas as pd
from datetime import datetime

user_rows = []

with open(r"C:\Users\lakshita rawat\Downloads\revenge\user.json") as f:
    users = json.load(f)

for user in users:   # ✅ correct loop

    uid = str(user.get("id"))

    created_at = user.get("created_at", None)

    # Convert to datetime
    try:
        created_at = datetime.strptime(created_at, "%a %b %d %H:%M:%S %z %Y")
        account_age_days = (datetime.now(created_at.tzinfo) - created_at).days
    except:
        account_age_days = 0

    user_rows.append({
        "id": uid,
        "followers_count": user.get("followers_count", 0),
        "friends_count": user.get("friends_count", 0),
        "statuses_count": user.get("statuses_count", 0),
        "favourites_count": user.get("favourites_count", 0),
        "verified": int(user.get("verified", False)),
        "default_profile_image": int(user.get("default_profile_image", False)),
        "account_age_days": account_age_days
    })

user_df = pd.DataFrame(user_rows)

print(user_df.head())
print(user_df.columns)

                     id  followers_count  friends_count  statuses_count  \
0  u1217628182611927040                0              0               0   
1           u2664730894                0              0               0   
2  u1266703520205549568                0              0               0   
3  u1089159225148882949                0              0               0   
4             u36741729                0              0               0   

   favourites_count  verified  default_profile_image  account_age_days  
0                 0         0                      0                 0  
1                 0         0                      0                 0  
2                 0         0                      0                 0  
3                 0         0                      0                 0  
4                 0         0                      0                 0  
Index(['id', 'followers_count', 'friends_count', 'statuses_count',
       'favourites_count', 'verified', 'defa

In [6]:
labels_df = pd.read_csv(r"C:\Users\lakshita rawat\Downloads\revenge\label.csv", dtype={"id": str})
split_df = pd.read_csv(r"C:\Users\lakshita rawat\Downloads\revenge\split.csv", dtype={"id": str})

In [7]:
df = labels_df.merge(split_df, on="id", how="inner")
df = df.merge(user_df, on="id", how="inner")

print(df.head())
print(df.columns)

                     id  label  split  followers_count  friends_count  \
0  u1217628182611927040  human   test                0              0   
1           u2664730894  human  train                0              0   
2  u1266703520205549568  human   test                0              0   
3  u1089159225148882949  human  train                0              0   
4             u36741729    bot  train                0              0   

   statuses_count  favourites_count  verified  default_profile_image  \
0               0                 0         0                      0   
1               0                 0         0                      0   
2               0                 0         0                      0   
3               0                 0         0                      0   
4               0                 0         0                      0   

   account_age_days  
0                 0  
1                 0  
2                 0  
3                 0  
4                 

In [8]:
df.fillna(0, inplace=True)

In [9]:
print(df.columns.tolist())

['id', 'label', 'split', 'followers_count', 'friends_count', 'statuses_count', 'favourites_count', 'verified', 'default_profile_image', 'account_age_days']


In [83]:
rows = []

for user in users:
    metrics = user.get("public_metrics", {})

    row = {}

    row["id"] = user.get("id", "")
    row["label"] = user.get("label", "")  # if exists

    # ✅ FIXED MAPPING
    row["followers_count"] = metrics.get("followers_count", 0)
    row["friends_count"]   = metrics.get("following_count", 0)
    row["statuses_count"]  = metrics.get("tweet_count", 0)
    row["favourites_count"] = 0  # not available → keep 0

    row["verified"] = int(user.get("verified", False))
    row["default_profile_image"] = 0  # not available → keep 0

    # Extra useful fields
    row["has_location"] = int(bool(user.get("location")))
    row["has_bio"] = int(bool(user.get("description")))
    row["pinned_tweet"] = int(user.get("pinned_tweet_id") is not None)

    # Account age
    from datetime import datetime
    created = user.get("created_at")

    try:
        created_date = datetime.strptime(created, "%Y-%m-%d %H:%M:%S%z")
        age_days = (datetime.now(created_date.tzinfo) - created_date).days
    except:
        age_days = 365

    row["account_age_days"] = age_days

    rows.append(row)

df = pd.DataFrame(rows)

In [84]:
print(df.head())

                     id label  followers_count  friends_count  statuses_count  \
0  u1217628182611927040                   7316            215            3098   
1           u2664730894                    123           1090            1823   
2  u1266703520205549568                      3             62              66   
3  u1089159225148882949                    350            577             237   
4             u36741729                    240            297            3713   

   favourites_count  verified  default_profile_image  has_location  has_bio  \
0                 0         0                      0             1        1   
1                 0         0                      0             1        1   
2                 0         0                      0             0        1   
3                 0         0                      0             1        1   
4                 0         0                      0             1        1   

   pinned_tweet  account_age_days  
0 

In [85]:
import numpy as np

df["followers_friends_ratio"] = df["followers_count"] / (df["friends_count"] + 1)
df["statuses_per_day"] = df["statuses_count"] / (df["account_age_days"] + 1)
df["favourites_per_status"] = df["favourites_count"] / (df["statuses_count"] + 1)

df["log_followers"] = np.log1p(df["followers_count"])
df["log_friends"]   = np.log1p(df["friends_count"])
df["log_statuses"]  = np.log1p(df["statuses_count"])

In [86]:
FEATURES = [
    "followers_count",
    "friends_count",
    "statuses_count",
    "favourites_count",
    "verified",
    "default_profile_image",
    "account_age_days",
    "followers_friends_ratio",
    "statuses_per_day",
    "favourites_per_status",
    "log_followers",
    "log_friends",
    "log_statuses"
]

X = df[FEATURES]
y = df["label"]

In [64]:
print(df.columns)

Index(['id', 'label', 'followers_count', 'friends_count', 'statuses_count',
       'favourites_count', 'verified', 'default_profile_image', 'has_location',
       'has_bio', 'pinned_tweet', 'account_age_days',
       'followers_friends_ratio', 'statuses_per_day', 'favourites_per_status',
       'log_followers', 'log_friends', 'log_statuses'],
      dtype='object')


In [57]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [58]:
print("Train:")
print(y_train.value_counts())

print("\nTest:")
print(y_test.value_counts())

Train:
label
    800000
Name: count, dtype: int64

Test:
label
    200000
Name: count, dtype: int64


In [59]:
y_train = y_train.map({"human": 0, "bot": 1})
y_test = y_test.map({"human": 0, "bot": 1})

In [60]:
print(y_train.value_counts())
print(y_train.unique())

Series([], Name: count, dtype: int64)
[nan]


In [75]:
labels_df=df = pd.read_csv(r"C:\Users\lakshita rawat\Downloads\revenge\label.csv")
print(labels_df.head())

                     id  label
0  u1217628182611927040  human
1           u2664730894  human
2  u1266703520205549568  human
3  u1089159225148882949  human
4             u36741729    bot


In [78]:
print(labels_df.columns)

Index(['id', 'label'], dtype='object')


In [79]:
labels_df = labels_df.rename(columns={
    "user_id": "id",
    "account_type": "label"
})

In [80]:
print(df.columns)

Index(['id', 'label_x', 'label_y'], dtype='object')


In [81]:
df = df.merge(labels_df, on="id", how="left")

In [93]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [46]:
import pandas as pd

train_df = pd.concat([X_train, y_train], axis=1)

In [65]:
train_df = pd.concat([X_train, y_train], axis=1)
print(train_df["label"].value_counts())

label
    800000
Name: count, dtype: int64


In [95]:
print(df["label"].value_counts())

label
    1000000
Name: count, dtype: int64


In [89]:
print(labels_df.columns)
print(labels_df.head())

Index(['id', 'label'], dtype='object')
                     id  label
0  u1217628182611927040  human
1           u2664730894  human
2  u1266703520205549568  human
3  u1089159225148882949  human
4             u36741729    bot


In [90]:
print(df.columns)

Index(['id', 'label', 'followers_count', 'friends_count', 'statuses_count',
       'favourites_count', 'verified', 'default_profile_image', 'has_location',
       'has_bio', 'pinned_tweet', 'account_age_days',
       'followers_friends_ratio', 'statuses_per_day', 'favourites_per_status',
       'log_followers', 'log_friends', 'log_statuses'],
      dtype='object')


In [67]:
print(df.columns)
print(df.head())

Index(['id', 'label', 'followers_count', 'friends_count', 'statuses_count',
       'favourites_count', 'verified', 'default_profile_image', 'has_location',
       'has_bio', 'pinned_tweet', 'account_age_days',
       'followers_friends_ratio', 'statuses_per_day', 'favourites_per_status',
       'log_followers', 'log_friends', 'log_statuses'],
      dtype='object')
                     id label  followers_count  friends_count  statuses_count  \
0  u1217628182611927040                   7316            215            3098   
1           u2664730894                    123           1090            1823   
2  u1266703520205549568                      3             62              66   
3  u1089159225148882949                    350            577             237   
4             u36741729                    240            297            3713   

   favourites_count  verified  default_profile_image  has_location  has_bio  \
0                 0         0                      0             1 

In [96]:
print("FULL DATA LABELS:")
print(df["label"].value_counts())

print("\nTRAIN LABELS:")
print(train_df["label"].value_counts())

FULL DATA LABELS:
label
    1000000
Name: count, dtype: int64

TRAIN LABELS:
label
    800000
Name: count, dtype: int64


In [97]:
print(df["id"].iloc[0])
print(labels_df["id"].iloc[0])

u1217628182611927040
u1217628182611927040


In [98]:
df["id"] = df["id"].astype(str).str.strip()
labels_df["id"] = labels_df["id"].astype(str).str.strip()

In [99]:
label_map = dict(zip(labels_df["id"], labels_df["label"]))
df["label"] = df["id"].map(label_map)

In [100]:
print(df["label"].value_counts())
print("Missing labels:", df["label"].isna().sum())

label
human    860057
bot      139943
Name: count, dtype: int64
Missing labels: 0


In [102]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [103]:
import pandas as pd

train_df = pd.concat([X_train, y_train], axis=1)

print(train_df["label"].value_counts())

label
human    688046
bot      111954
Name: count, dtype: int64


In [104]:
from sklearn.utils import resample

df_majority = train_df[train_df["label"] == "human"]
df_minority = train_df[train_df["label"] == "bot"]

df_majority_under = resample(
    df_majority,
    replace=False,
    n_samples=len(df_minority),
    random_state=42
)

df_under = pd.concat([df_majority_under, df_minority])
df_under = df_under.sample(frac=1, random_state=42)

print(df_under["label"].value_counts())

label
human    111954
bot      111954
Name: count, dtype: int64


In [109]:
from xgboost import XGBClassifier

X_train_under = df_under[FEATURES]
y_train_under = df_under["label"].map({"human": 0, "bot": 1})

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train_under, y_train_under)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [110]:
y_test_encoded = y_test.map({"human": 0, "bot": 1})

y_pred = model.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(y_test_encoded, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.71      0.82    172011
           1       0.31      0.80      0.45     27989

    accuracy                           0.72    200000
   macro avg       0.63      0.76      0.63    200000
weighted avg       0.87      0.72      0.76    200000



### UNDERSAMPLING

In [111]:
y_prob = xgb.predict_proba(X_test)[:, 1]

In [112]:
feat_imp = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": xgb.feature_importances_
}).sort_values(by="Importance", ascending=False)

print(feat_imp)

                    Feature  Importance
4                  verified         1.0
0           followers_count         0.0
1             friends_count         0.0
2            statuses_count         0.0
3          favourites_count         0.0
5     default_profile_image         0.0
6          account_age_days         0.0
7   followers_friends_ratio         0.0
8          statuses_per_day         0.0
9     favourites_per_status         0.0
10            log_followers         0.0
11              log_friends         0.0
12             log_statuses         0.0


In [113]:
import joblib
import json

model=joblib.dump(xgb, 'xgboost_model.pkl')

feature_cols = list(X_train_under.columns)

with open('feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

print("Model saved successfully!")

Model saved successfully!


In [115]:
import json

feature_cols = list(X_train_under.columns)

with open("feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("Feature columns saved!")

Feature columns saved!


In [114]:
print(xgb.predict_proba(X_test[:10]))

[[4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]
 [9.9990278e-01 9.7196942e-05]
 [4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]
 [4.7059917e-01 5.2940083e-01]]


In [24]:
print(X_train.columns)

Index(['followers_count', 'friends_count', 'statuses_count',
       'favourites_count', 'verified', 'default_profile_image',
       'account_age_days', 'followers_friends_ratio', 'statuses_per_day',
       'favourites_per_status', 'log_followers', 'log_friends',
       'log_statuses'],
      dtype='object')


In [25]:
print(X_train.head())

        followers_count  friends_count  statuses_count  favourites_count  \
475827                0              0               0                 0   
404873                0              0               0                 0   
806862                0              0               0                 0   
665707                0              0               0                 0   
90060                 0              0               0                 0   

        verified  default_profile_image  account_age_days  \
475827         0                      0                 0   
404873         1                      0                 0   
806862         0                      0                 0   
665707         0                      0                 0   
90060          1                      0                 0   

        followers_friends_ratio  statuses_per_day  favourites_per_status  \
475827                      0.0               0.0                    0.0   
404873                  

In [26]:
print(df.head())

                     id  label  split  followers_count  friends_count  \
0  u1217628182611927040  human   test                0              0   
1           u2664730894  human  train                0              0   
2  u1266703520205549568  human   test                0              0   
3  u1089159225148882949  human  train                0              0   
4             u36741729    bot  train                0              0   

   statuses_count  favourites_count  verified  default_profile_image  \
0               0                 0         0                      0   
1               0                 0         0                      0   
2               0                 0         0                      0   
3               0                 0         0                      0   
4               0                 0         0                      0   

   account_age_days  followers_friends_ratio  statuses_per_day  \
0                 0                      0.0               0.0

In [27]:
print(users[0])

{'created_at': '2020-01-16 02:02:55+00:00', 'description': 'Theoretical Computer Scientist. See also https://t.co/EXWR5jOrFW and https://t.co/JEkxX4JHSw', 'entities': {'url': {'urls': [{'start': 0, 'end': 23, 'url': 'https://t.co/BoMip9FF17', 'expanded_url': 'https://www.boazbarak.org/', 'display_url': 'boazbarak.org'}]}, 'description': {'urls': [{'start': 41, 'end': 64, 'url': 'https://t.co/EXWR5jOrFW', 'expanded_url': 'http://windowsontheory.org', 'display_url': 'windowsontheory.org'}, {'start': 69, 'end': 92, 'url': 'https://t.co/JEkxX4JHSw', 'expanded_url': 'http://mltheory.org', 'display_url': 'mltheory.org'}]}}, 'id': 'u1217628182611927040', 'location': 'Cambridge, MA', 'name': 'Boaz Barak', 'pinned_tweet_id': None, 'profile_image_url': 'https://pbs.twimg.com/profile_images/1252262363132280834/ytIN-vzv_normal.jpg', 'protected': False, 'public_metrics': {'followers_count': 7316, 'following_count': 215, 'tweet_count': 3098, 'listed_count': 69}, 'url': 'https://t.co/BoMip9FF17', 'us